# Breast Cancer RBF SVM

**Programmer:** Jeffrey Morales  
**Description:** Preprocesses the breast cancer dataset, tunes and evaluates an RBF Support Vector Machine, selects a screening threshold, and calibrates predicted probabilities.

In [2]:
from pathlib import Path
import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

print(f"Python: {platform.python_version()}")
print(f"scikit-learn: {sklearn.__version__}")

Python: 3.13.5
scikit-learn: 1.6.1


In [3]:
# Find the dataset in the project or uploaded workspace
candidate_paths = [
    Path("breast_cancer_prediction.csv"),
    Path("upload/breast_cancer_prediction.csv"),
    Path("/workspace/scratch/57ca16fe0d7b/upload/breast_cancer_prediction.csv"),
]
DATA_PATH = next((path for path in candidate_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find breast_cancer_prediction.csv. "
        "Place it in the same folder as this notebook."
    )

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH.resolve()}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

FileNotFoundError: Could not find breast_cancer_prediction.csv. Place it in the same folder as this notebook.

In [ ]:
# Check class balance before modeling
target_summary = (
    df["Cancer"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("Cancer")
    .to_frame("Count")
)
target_summary["Percent"] = 100 * target_summary["Count"] / len(df)
display(target_summary)
print(f"Majority-class accuracy baseline: {(df['Cancer'] == 0).mean():.3f}")

In [ ]:
# List columns that need missing-value handling
missing_summary = (
    df.isna().sum()
    .rename("Missing")
    .to_frame()
    .assign(Percent=lambda table: 100 * table["Missing"] / len(df))
    .query("Missing > 0")
    .sort_values("Missing", ascending=False)
)
missing_summary

In [ ]:
# Remove identifiers and post-diagnosis leakage fields
TARGET = "Cancer"
EXCLUDED_COLUMNS = [
    "Patient_ID",
    TARGET,
    "Biopsy_Result",
    "Cancer_Stage",
    "Mammogram_Result",
    "Tumor_Size_cm",
    "Lymph_Node_Involvement",
]

X = df.drop(columns=EXCLUDED_COLUMNS)
y = df[TARGET].astype(int)

print(f"Predictors retained: {X.shape[1]}")
display(pd.DataFrame({"Predictor": X.columns, "Type": X.dtypes.astype(str).values}))

In [ ]:
numeric_eda_columns = X.select_dtypes(include=np.number).columns.tolist()
eda_data = X[numeric_eda_columns].copy()
eda_data[TARGET] = y

n_columns = 3
n_rows = int(np.ceil(len(numeric_eda_columns) / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(16, 4.5 * n_rows))
axes = np.atleast_1d(axes).ravel()

for ax, column in zip(axes, numeric_eda_columns):
    sns.histplot(
        data=eda_data,
        x=column,
        hue=TARGET,
        bins=30,
        stat="density",
        common_norm=False,
        element="step",
        fill=False,
        ax=ax,
    )
    ax.set_title(f"{column} by cancer status")

for ax in axes[len(numeric_eda_columns):]:
    ax.remove()

fig.suptitle("Numerical predictor distributions by target class", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
male_quality_check = (
    df.loc[df["Gender"].eq("Male"), ["Menopause_Status", "Breastfeeding_History"]]
    .value_counts()
    .rename("Records")
    .reset_index()
)
male_quality_check.head(10)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame(
    {
        "Rows": [len(y_train), len(y_test)],
        "Cancer prevalence": [y_train.mean(), y_test.mean()],
    },
    index=["Training", "Test"],
)
split_summary

In [ ]:
numeric_columns = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_columns = [
    column for column in X_train.columns if column not in numeric_columns
]

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns),
    ]
)

rbf_svm_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            SVC(
                kernel="rbf",
                class_weight="balanced",
                cache_size=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

print("Numeric predictors:", numeric_columns)
print("Categorical predictors:", categorical_columns)

In [ ]:
# Tune regularization strength and kernel width
parameter_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", 0.01, 0.1],
}

inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

search = GridSearchCV(
    estimator=rbf_svm_pipeline,
    param_grid=parameter_grid,
    scoring="average_precision",
    cv=inner_cv,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
)
search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print(f"Best mean validation PR-AUC: {search.best_score_:.4f}")

In [ ]:
cv_results = (
    pd.DataFrame(search.cv_results_)
    .loc[
        :,
        [
            "param_model__C",
            "param_model__gamma",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "rank_test_score",
        ],
    ]
    .sort_values("rank_test_score")
    .rename(
        columns={
            "param_model__C": "C",
            "param_model__gamma": "gamma",
            "mean_test_score": "Mean validation PR-AUC",
            "std_test_score": "Validation SD",
            "mean_train_score": "Mean training PR-AUC",
            "rank_test_score": "Rank",
        }
    )
)
cv_results.head(12)

In [ ]:
# Evaluate the best model once on the held-out test set
best_model = search.best_estimator_
test_predictions = best_model.predict(X_test)
test_scores = best_model.decision_function(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, test_predictions).ravel()
test_metrics = pd.Series(
    {
        "ROC-AUC": roc_auc_score(y_test, test_scores),
        "PR-AUC": average_precision_score(y_test, test_scores),
        "Accuracy (context only)": accuracy_score(y_test, test_predictions),
        "Balanced accuracy": balanced_accuracy_score(y_test, test_predictions),
        "Sensitivity / recall": recall_score(y_test, test_predictions),
        "Specificity": tn / (tn + fp),
        "Positive predictive value / precision": precision_score(y_test, test_predictions),
        "Negative predictive value": tn / (tn + fn),
        "False-negative rate": fn / (fn + tp),
        "F1 score": f1_score(y_test, test_predictions),
        "Matthews correlation coefficient": matthews_corrcoef(
            y_test, test_predictions
        ),
    },
    name="Test value",
)
display(test_metrics.to_frame().round(4))
print(classification_report(y_test, test_predictions, target_names=["No cancer", "Cancer"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_predictions,
    display_labels=["No cancer", "Cancer"],
    cmap="Blues",
    colorbar=False,
    ax=axes[0],
)
axes[0].set_title("Confusion matrix")

RocCurveDisplay.from_predictions(y_test, test_scores, ax=axes[1], color="#4C78A8")
axes[1].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
axes[1].set_title("ROC curve")

PrecisionRecallDisplay.from_predictions(
    y_test, test_scores, ax=axes[2], color="#E45756"
)
axes[2].axhline(y_test.mean(), linestyle="--", color="gray", linewidth=1)
axes[2].set_title("Precision–recall curve")

plt.tight_layout()
plt.show()

In [ ]:
# Estimate uncertainty around the test metrics
def bootstrap_metric_intervals(
    y_true, predictions, scores, n_bootstrap=1_000, random_state=RANDOM_STATE
):
    y_array = np.asarray(y_true)
    prediction_array = np.asarray(predictions)
    score_array = np.asarray(scores)
    rng = np.random.default_rng(random_state)
    bootstrap_rows = []

    for _ in range(n_bootstrap):
        indices = rng.integers(0, len(y_array), size=len(y_array))
        y_sample = y_array[indices]
        if np.unique(y_sample).size < 2:
            continue

        prediction_sample = prediction_array[indices]
        score_sample = score_array[indices]
        tn_b, fp_b, fn_b, tp_b = confusion_matrix(
            y_sample, prediction_sample, labels=[0, 1]
        ).ravel()
        bootstrap_rows.append(
            {
                "ROC-AUC": roc_auc_score(y_sample, score_sample),
                "PR-AUC": average_precision_score(y_sample, score_sample),
                "Sensitivity": tp_b / (tp_b + fn_b),
                "Specificity": tn_b / (tn_b + fp_b),
                "Negative predictive value": tn_b / (tn_b + fn_b),
                "Matthews correlation coefficient": matthews_corrcoef(
                    y_sample, prediction_sample
                ),
            }
        )

    bootstrap_results = pd.DataFrame(bootstrap_rows)
    return pd.DataFrame(
        {
            "Estimate": bootstrap_results.mean(),
            "95% CI lower": bootstrap_results.quantile(0.025),
            "95% CI upper": bootstrap_results.quantile(0.975),
        }
    )

bootstrap_intervals = bootstrap_metric_intervals(
    y_test, test_predictions, test_scores
)
bootstrap_intervals.round(4)

In [ ]:
# Pick a screening threshold from training predictions only
threshold_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE + 1,
)
oof_training_scores = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=threshold_cv,
    method="decision_function",
    n_jobs=-1,
)

TARGET_SENSITIVITY = 0.90
fpr, tpr, thresholds = roc_curve(y_train, oof_training_scores)
eligible = np.flatnonzero(tpr >= TARGET_SENSITIVITY)
if len(eligible) == 0:
    raise RuntimeError("No threshold achieved the requested sensitivity.")
threshold_index = eligible[np.argmax(1 - fpr[eligible])]
screening_threshold = thresholds[threshold_index]

screening_predictions = (test_scores >= screening_threshold).astype(int)

def threshold_metrics(y_true, predictions):
    tn, fp, fn, tp = confusion_matrix(y_true, predictions).ravel()
    return {
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions),
        "Predicted positive": int(predictions.sum()),
    }

threshold_comparison = pd.DataFrame(
    {
        "Default SVM boundary": threshold_metrics(y_test, test_predictions),
        f"Training-selected ≥{TARGET_SENSITIVITY:.0%} sensitivity": threshold_metrics(
            y_test, screening_predictions
        ),
    }
).T

print(f"Training-selected decision threshold: {screening_threshold:.4f}")
threshold_comparison

In [ ]:
# Convert SVM decision scores into calibrated probabilities
calibration_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE + 2,
)
calibrated_model = CalibratedClassifierCV(
    estimator=best_model,
    method="sigmoid",
    cv=calibration_cv,
    n_jobs=-1,
)
calibrated_model.fit(X_train, y_train)
test_probabilities = calibrated_model.predict_proba(X_test)[:, 1]

probability_metrics = pd.Series(
    {
        "ROC-AUC": roc_auc_score(y_test, test_probabilities),
        "PR-AUC": average_precision_score(y_test, test_probabilities),
        "Brier score (lower is better)": brier_score_loss(
            y_test, test_probabilities
        ),
    },
    name="Calibrated test value",
)
probability_metrics.to_frame().round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
CalibrationDisplay.from_predictions(
    y_test,
    test_probabilities,
    n_bins=10,
    strategy="quantile",
    ax=ax,
    color="#4C78A8",
)
ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
ax.set_title("Calibration of RBF SVM probabilities")
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

teal_cmap = LinearSegmentedColormap.from_list("teal", ["#EAF5F5", "#7CC4C0", "#0E7C86", "#0E2A33"])
DK, TEAL, GOLD, CORAL = "#0E2A33", "#0E7C86", "#F2C14E", "#E8825A"

In [ ]:
# Toy example showing how an RBF kernel creates a curved boundary
rng = np.random.default_rng(3); n = 90
a0 = rng.uniform(0, 2*np.pi, n); r0 = rng.normal(0.0, 0.9, n)
a1 = rng.uniform(0, 2*np.pi, n); r1 = rng.normal(3.4, 0.5, n)
X0 = np.c_[r0*np.cos(a0), r0*np.sin(a0)]      # center blob
X1 = np.c_[r1*np.cos(a1), r1*np.sin(a1)]      # surrounding ring
X = np.vstack([X0, X1]); y = np.r_[np.zeros(n), np.ones(n)]

clf = SVC(kernel="rbf", C=10, gamma=0.3).fit(X, y)
xx, yy = np.meshgrid(np.linspace(-5, 5, 400), np.linspace(-5, 5, 400))
Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(6.6, 5.4))
ax.contourf(xx, yy, Z, levels=[-1e9, 0, 1e9], colors=["#D6ECEC", "#FBE6D3"], alpha=0.9)
ax.contour(xx, yy, Z, levels=[0], colors=[DK], linewidths=2.6)                    # boundary
ax.contour(xx, yy, Z, levels=[-1, 1], colors=[TEAL, CORAL], linewidths=1.2, linestyles="--")  # margins
ax.scatter(*X0.T, c=TEAL, s=34, edgecolor="white", lw=0.5, label="Class A", zorder=3)
ax.scatter(*X1.T, c=CORAL, s=34, edgecolor="white", lw=0.5, label="Class B", zorder=3)
sv = clf.support_vectors_
ax.scatter(*sv.T, s=150, facecolors="none", edgecolors=GOLD, linewidths=2.2, label="Support vectors", zorder=2)
ax.set_xlim(-5, 5); ax.set_ylim(-5, 5); ax.set_xticks([]); ax.set_yticks([])
ax.legend(loc="upper right", fontsize=10, framealpha=0.95)
ax.set_title("RBF kernel bends the boundary to separate\nclasses a straight line never could", color=DK, pad=10)
plt.tight_layout(); plt.show()